In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Day 05.4 · Cierre operativo y resumen ejecutivo antes de producto

**Objetivo de este notebook**: dejar un resumen único, legible y útil para presentación de todo lo aprendido en `Notebook 19` y `Notebook 20`, sin reabrir nuevas iteraciones Day 05.4x.

La lectura oficial del bloque queda así:
- `Notebook 19` explica el champion puro vigente y descubre slices operativos defendibles.
- `Notebook 20` conserva la auditoría técnica completa de `Day 05.4`, `Day 05.4b` y `Day 05.4c`.
- este `Notebook 21` resume qué conseguimos de verdad y por qué el bloque se cierra con `keep_current_operational_policy`.


## 1. Qué buscábamos con Error Analysis + SHAP

Este bloque no entró para entrenar otro modelo ni para abrir Brent. Entró para responder una pregunta mucho más operativa:

**¿podemos entender mejor el champion puro y capturar mejoras locales reproducibles sin tocar pesos ni features?**

En este proyecto, `Error Analysis` sirvió para localizar **dónde falla** el champion puro sobre el holdout oficial. `SHAP` sirvió para explicar **por qué** esos slices parecían distintos y si esa diferencia era traducible a una regla observable en inferencia.

El valor real del bloque fue doble:
- mejorar criterio sobre el modelo actual;
- comprobar si algunas ganancias locales podían convertirse en policies postinferencia auditables.


## 2. Qué explicó Notebook 19 del champion puro

`Notebook 19` dejó cuatro conclusiones operativas claras:

1. `PRODUCT_005` dejaba un fallback local serio y relativamente limpio.
2. La slice dominante de `PRODUCT_003` aportaba una mejora de `Top-2` y una señal de baja confianza, pero no siempre cambio real de `decision_final`.
3. `PRODUCT_003 residual` sí dejaba complementariedad útil, pero mezclaba wins con daño y por tanto necesitaba cortes mucho más finos.
4. `both_fail_top2` apuntaba más a falta de señal que a una regla simple implementable.

La consecuencia práctica fue importante: el notebook no solo explicó el champion, también nos dijo **qué merecía implementación** y **qué no**.


In [2]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT

REGISTRY_PATH = PROJECT_ROOT / "artifacts" / "public" / "metrics" / "final_baseline_vs_candidates.csv"
DAY054_RUN_ID = "20260310T110104Z"
DAY054B_RUN_ID = "20260310T132832Z"
DAY054C_RUN_ID = "20260310T140709Z"

registry_df = pd.read_csv(REGISTRY_PATH)
day054_summary = json.loads((PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / f"{DAY054_RUN_ID}_run_summary.json").read_text(encoding="utf-8"))
day054b_summary = json.loads((PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / f"{DAY054B_RUN_ID}_run_summary.json").read_text(encoding="utf-8"))
day054c_summary = json.loads((PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / f"{DAY054C_RUN_ID}_run_summary.json").read_text(encoding="utf-8"))
day054_trials = pd.read_csv(PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / f"{DAY054_RUN_ID}_policy_trials.csv")
day054b_trials = pd.read_csv(PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / f"{DAY054B_RUN_ID}_policy_trials.csv")
day054c_trials = pd.read_csv(PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_4" / f"{DAY054C_RUN_ID}_policy_trials.csv")

reference_specs = [
    ("champion_pure", "V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1", "modelo puro vigente"),
    ("serving_default", "LR_smote_0.5", "default congelado por despliegue"),
    ("best_operational_policy", "V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1", "policy operativa vigente en gobernanza"),
]
reference_rows = []
for role, variant, status in reference_specs:
    ref_row = registry_df.loc[registry_df["model_variant"].eq(variant)].iloc[-1]
    reference_rows.append(
        {
            "reference_role": role,
            "variant": variant,
            "status": status,
            "top1_hit": float(ref_row["top1_hit"]),
            "top2_hit": float(ref_row["top2_hit"]),
        }
    )
references_df = pd.DataFrame(reference_rows)
references_df[["top1_hit", "top2_hit"]] = references_df[["top1_hit", "top2_hit"]].round(6)
display(references_df)


,reference_role,variant,status,top1_hit,top2_hit
0,champion_pure,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,modelo puro vigente,0.772503,0.882643
1,serving_default,LR_smote_0.5,default congelado por despliegue,0.764527,0.858336
2,best_operational_policy,V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINIS...,policy operativa vigente en gobernanza,0.789594,0.860235


## 3. Qué policies candidatas nacieron de ese análisis

La capa experimental Day 05.4x no intentó inventar familias nuevas de señal. Se limitó a operacionalizar exactamente lo descubierto en `Notebook 19`:

- `PRODUCT_005` como fallback local candidato para rescatar `Top-1`.
- `PRODUCT_003 dominante` como mejora de `Top-2`/ranking y señal de baja confianza.
- `PRODUCT_003 residual` como línea de complementariedad a limpiar con reglas más finas.

La lógica de trabajo fue siempre la misma:
- reproducir baseline, champion y policy operativa sobre el mismo holdout oficial;
- activar una regla observable y trazable;
- medir ayuda/daño;
- decidir si la mejora era suficientemente material y segura como para pasar el gate.


## 4. Qué se implementó y auditó en Notebook 20

`Notebook 20` se convirtió en el audit trail del bloque y dejó tres capas de evidencia:

1. `Day 05.4`: implementación modular inicial y posterior rerun limpio tras invalidar el scorer roto.
2. `Day 05.4b`: microiteración residual/composite para intentar ganar a la policy operativa vigente.
3. `Day 05.4c`: endurecimiento final para eliminar `harmed` sin reabrir gobernanza.

Ese notebook ya no es el sitio para contar la historia corta. Su papel es conservar la trazabilidad técnica completa: runs, artifacts, smokes, guardrails, invalidación y reconciliación final.


In [3]:
day054_composite_row = day054_trials.loc[
    day054_trials["policy_variant"].eq("day054_composite_go_c_plus_go_b_to_baseline_v1")
].iloc[0]
day054b_composite_row = day054b_trials.loc[
    day054b_trials["policy_variant"].eq("day054b_composite_go_c_plus_go_b_dominant_plus_residual_precision_v1")
].iloc[0]
day054c_composite_row = day054c_trials.loc[
    day054c_trials["policy_variant"].eq("day054c_composite_go_c_plus_go_b_dominant_plus_residual_rank2_precision_v1")
].iloc[0]

run_summary_df = pd.DataFrame([
    {
        "day_block": "Day 05.4",
        "run_id": DAY054_RUN_ID,
        "policy_finalista": str(day054_composite_row["policy_variant"]),
        "top1_hit": float(day054_composite_row["top1_hit"]),
        "top2_hit": float(day054_composite_row["top2_hit"]),
        "overrides_harmed": int(day054_composite_row["overrides_harmed"]),
        "failure_reason": str(day054_composite_row["failure_reason"]),
        "decision": day054_summary["decision"],
    },
    {
        "day_block": "Day 05.4b",
        "run_id": DAY054B_RUN_ID,
        "policy_finalista": str(day054b_composite_row["policy_variant"]),
        "top1_hit": float(day054b_composite_row["top1_hit"]),
        "top2_hit": float(day054b_composite_row["top2_hit"]),
        "overrides_harmed": int(day054b_composite_row["overrides_harmed"]),
        "failure_reason": str(day054b_composite_row["failure_reason"]),
        "decision": day054b_summary["decision"],
    },
    {
        "day_block": "Day 05.4c",
        "run_id": DAY054C_RUN_ID,
        "policy_finalista": str(day054c_composite_row["policy_variant"]),
        "top1_hit": float(day054c_composite_row["top1_hit"]),
        "top2_hit": float(day054c_composite_row["top2_hit"]),
        "overrides_harmed": int(day054c_composite_row["overrides_harmed"]),
        "failure_reason": str(day054c_composite_row["failure_reason"]),
        "decision": day054c_summary["decision"],
    },
])
run_summary_df[["top1_hit", "top2_hit"]] = run_summary_df[["top1_hit", "top2_hit"]].round(6)
display(run_summary_df)


,day_block,run_id,policy_finalista,top1_hit,top2_hit,overrides_harmed,failure_reason,decision
0,Day 05.4,20260310T110104Z,day054_composite_go_c_plus_go_b_to_baseline_v1,0.779719,0.889859,0,operational_gate_fail,keep_current_operational_policy
1,Day 05.4b,20260310T132832Z,day054b_composite_go_c_plus_go_b_dominant_plus...,0.791493,0.890619,7,champion_guardrail_fail,keep_current_operational_policy
2,Day 05.4c,20260310T140709Z,day054c_composite_go_c_plus_go_b_dominant_plus...,0.786935,0.890619,0,operational_gate_fail,keep_current_operational_policy


## 5. Qué mejoras reales conseguimos

La conclusión correcta no es “no conseguimos nada”. Sí conseguimos varias cosas valiosas:

- explicar mucho mejor el champion puro vigente;
- demostrar que había mejoras locales reales frente al champion;
- comprobar que algunas ganancias en `Top-2` o en slices concretos sí eran capturables;
- descubrir que parte de esas ganancias se pagaban con daño local o con pérdida de `Top-1` frente a la mejor policy operativa vigente.

Dicho de forma práctica: el bloque sirvió para **entender** y para **mejorar parcialmente**, pero no para encontrar una nueva policy operativa que superara limpiamente a la vigente.


In [4]:
comparison_df = pd.DataFrame([
    {
        "candidate": "Day 05.4b composite",
        "top1_hit": float(day054b_composite_row["top1_hit"]),
        "top2_hit": float(day054b_composite_row["top2_hit"]),
        "delta_top1_vs_operational": float(day054b_composite_row["delta_top1_vs_operational"]),
        "delta_top2_vs_operational": float(day054b_composite_row["delta_top2_vs_operational"]),
        "overrides_harmed": int(day054b_composite_row["overrides_harmed"]),
        "reading": "+Top-1 y +Top-2, pero con daño local",
    },
    {
        "candidate": "Day 05.4c composite",
        "top1_hit": float(day054c_composite_row["top1_hit"]),
        "top2_hit": float(day054c_composite_row["top2_hit"]),
        "delta_top1_vs_operational": float(day054c_composite_row["delta_top1_vs_operational"]),
        "delta_top2_vs_operational": float(day054c_composite_row["delta_top2_vs_operational"]),
        "overrides_harmed": int(day054c_composite_row["overrides_harmed"]),
        "reading": "0 harmed y +Top-2, pero ya no supera a la policy operativa en Top-1",
    },
])
comparison_df[["top1_hit", "top2_hit", "delta_top1_vs_operational", "delta_top2_vs_operational"]] = comparison_df[["top1_hit", "top2_hit", "delta_top1_vs_operational", "delta_top2_vs_operational"]].round(6)
display(comparison_df)


,candidate,top1_hit,top2_hit,delta_top1_vs_operational,delta_top2_vs_operational,overrides_harmed,reading
0,Day 05.4b composite,0.791493,0.890619,0.001899,0.030384,7,"+Top-1 y +Top-2, pero con daño local"
1,Day 05.4c composite,0.786935,0.890619,-0.002659,0.030384,0,"0 harmed y +Top-2, pero ya no supera a la poli..."


## 6. Por qué ninguna policy pasó el gate operativo

El gate no comparaba contra el serving default congelado. Comparaba contra la **mejor policy operativa vigente cerrada en gobernanza**.

Eso obligaba a cumplir dos cosas a la vez:
- mejora material frente a esa policy operativa;
- guardrails sin daño local relevante.

Lo que vimos fue un tradeoff muy claro:
- `Day 05.4b` conseguía una compuesta fuerte, incluso mejor en agregado, pero con `7 harmed`.
- `Day 05.4c` limpiaba esos `harmed`, pero al hacerlo perdía suficiente `Top-1` como para quedar por debajo de la policy operativa vigente.

Por tanto, el bloque no deja una nueva policy promovible. Deja algo más importante para el proyecto final: una explicación seria del modelo actual, evidencia de mejoras locales reales y una frontera muy clara de qué no compensa promocionar.


## 7. Qué decide el proyecto ahora

La decisión final del bloque es única y explícita: `keep_current_operational_policy`.

Eso significa:
- se mantiene el champion puro vigente `V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1` como referencia de modelo puro;
- se mantiene el serving default congelado `LR_smote_0.5` por decisión de despliegue;
- se mantiene como mejor policy operativa vigente `V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1`;
- el bloque Day 05.4 queda cerrado documental y operativamente.


## 8. Siguiente foco: producto; Brent a backlog postbootcamp

La conclusión estratégica del proyecto cambia aquí.

Después de `Notebook 19`, `Notebook 20` y este cierre en `Notebook 21`, ya no tiene sentido seguir exprimiendo Day 05.4x dentro del bootcamp. El valor inmediato está en producto:
- `streamlit`
- `tableau`
- deploy
- presentación final

Brent no desaparece de la historia del proyecto. Queda aparcado como **backlog postbootcamp**: una línea futura de señal exógena/what-if que podrá retomarse después, pero que ya no gobierna el cierre del MVP ni la demo final.
